# MolmoE-1B -- Positional Bias Pipeline

Same pipeline structure as the Kosmos-2 / PaliGemma 2 / Florence-2 notebooks
(mount Drive, load samples, run zero-shot, save resumable per-dataset
results), adapted for `allenai/MolmoE-1B-0924` -- the mixture-of-experts
member of the Molmo family (1B active params per token, ~7B total,
Apache-2.0). It shares Molmo-7B-D's exact API and training data (PixMo),
so it natively answers "Point to X" with
`<point x=".." y="..">X</point>` tags giving a single (x, y) location as a
percentage of image width/height -- it does NOT emit bounding boxes like
Kosmos-2, PaliGemma, or Florence-2 do.

Because of that, this notebook scores MolmoE at the point/centroid level
only: `pred_bbox_*` columns are always left blank in the results CSV, and
`pred_centroid_x/y` comes directly from the parsed point (converted from
percentage to pixel coordinates using the original image size).

MolmoE-1B-0924 is Apache-2.0 licensed and NOT gated on Hugging Face, so no
HF token/login is required to download it (same as Kosmos-2). Its active
parameter count is much smaller than Molmo-7B-D's, which is friendlier to
a T4's 16GB -- dtype is pinned to bfloat16 below rather than left as
"auto" to avoid an accidental fp32 OOM.

In [ ]:
# Mount Google Drive to access files
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path

# Define the root directory for sample datasets on Google Drive
SAMPLES_ROOT = Path("/content/drive/MyDrive/samples")

# Define the root directory where results will be saved
RESULTS_ROOT = Path("/content/drive/MyDrive/results/molmoe")

# Create the results directory if it doesn't exist
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

print("Samples root:", SAMPLES_ROOT)
print("Samples root exists:", SAMPLES_ROOT.exists())

print("\nAvailable datasets:")
for folder in SAMPLES_ROOT.iterdir():
    if folder.is_dir():
        print("-", folder.name)


In [ ]:
# Molmo-specific deps on top of what Colab ships with -- einops and an
# up-to-date torchvision are required by Molmo's modeling code.
!pip install -U transformers accelerate einops torchvision


In [ ]:
import torch

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: No GPU detected!")


In [ ]:
import csv
import os
import re
from pathlib import Path

from PIL import Image

import torch
from transformers import AutoModelForCausalLM, AutoProcessor, GenerationConfig

# Define the model ID for MolmoE
MODEL_ID = "allenai/MolmoE-1B-0924"
# Define the datasets to be used
DATASETS = ("lgg", "kvasir", "isic")


In [ ]:
def to_molmo_prompt(prompt_text: str) -> str:
    """
    Rewrites the manifest's "Please locate ..." instruction into Molmo's
    documented pointing task format ("Point to X.") -- Molmo was trained on
    PixMo's pointing data with exactly this phrasing, and answers with
    <point x=".." y="..">X</point> (or <points ...> for multiple hits),
    same judgment call already made for Kosmos-2's phrase-grounding prompt:
    verify it actually respects the positional wording (left/right/upper/
    lower) rather than ignoring everything after the object name.

    'Please locate the tumor on the left side of this image.'
    -> 'Point to the tumor on the left side of this image.'
    """
    s = prompt_text.strip()
    if s.lower().startswith("please locate "):
        s = "Point to " + s[len("please locate "):]
    return s if s.endswith(".") else s + "."


_POINT_RE = re.compile(r'x="([\d.]+)"\s+y="([\d.]+)"')
_POINTS_RE = re.compile(r'x1="([\d.]+)"\s+y1="([\d.]+)"')


def parse_point_from_output(generated_text: str):
    """
    Molmo's pointing output looks like:
        <point x="49.3" y="52.1" alt="tumor">tumor</point>
    or, when it finds more than one instance:
        <points x1="10.0" y1="20.0" x2="30.0" y2="40.0" alt="tumor">tumor</points>

    Coordinates are percentages (0-100) of the ORIGINAL image's width/height,
    resolution independent. We only prompt for a single target, so this
    takes the FIRST point found (single-point tag first, multi-point tag as
    a fallback), matching the "expect one grounded entity" assumption used
    by parse_bbox_from_entities() in the Kosmos-2 notebook.

    Returns (x_pct, y_pct) or None if Molmo didn't emit a point at all
    (e.g. it just described the image in prose instead).
    """
    m = _POINT_RE.search(generated_text)
    if m:
        return float(m.group(1)), float(m.group(2))
    m = _POINTS_RE.search(generated_text)
    if m:
        return float(m.group(1)), float(m.group(2))
    return None


def point_pct_to_pixel_and_side(x_pct, y_pct, img_w, img_h):
    """
    Converts a percentage-of-image point to pixel coordinates and derives
    the same left/right, upper/lower side labels used everywhere else in
    this pipeline. Same convention as bbox_center_and_side() in the
    Kosmos-2 notebook, just starting from a point instead of a box.
    """
    cx = (x_pct / 100.0) * img_w
    cy = (y_pct / 100.0) * img_h
    side_lr = "left" if cx < img_w / 2 else "right"
    side_ud = "upper" if cy < img_h / 2 else "lower"
    return cx, cy, side_lr, side_ud


In [ ]:
def load_model():
    """
    Loads MolmoE and its processor. dtype is pinned to bfloat16 (rather than
    left as "auto", which can resolve to fp32 and roughly double memory use)
    since a T4's 16GB has little headroom for a ~7B-total MoE model; falls
    back to float32 on CPU, same pattern as the other model notebooks here.
    """
    dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

    processor = AutoProcessor.from_pretrained(
        MODEL_ID,
        trust_remote_code=True,
        dtype=dtype,
        device_map="auto",
    )

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        trust_remote_code=True,
        dtype=dtype,
        device_map="auto",
    )

    model.eval()
    return processor, model


In [ ]:
# Load the model and processor using the defined function
processor, model = load_model()

print("Model loaded successfully!")
print("Device:", model.device)


In [ ]:
def run_one(processor, model, image: Image.Image, prompt: str) -> str:
    """
    Runs Molmo on a single image with a single text prompt (the "Point to
    X." instruction built by to_molmo_prompt()).

    Uses autocast(bfloat16) on CUDA -- this is the setting Molmo's own
    model card evaluates in (autocast on, float32 weights) -- and generates
    until <|endoftext|> or 200 new tokens, same cap style as run_one() in
    the Kosmos-2 notebook.

    Returns the raw generated text (still containing the <point>/<points>
    tag, if any -- parsed separately by parse_point_from_output()).
    """
    inputs = processor.process(images=[image], text=prompt)
    inputs = {k: v.to(model.device).unsqueeze(0) for k, v in inputs.items()}

    with torch.autocast(device_type="cuda", enabled=torch.cuda.is_available(), dtype=torch.bfloat16):
        output = model.generate_from_batch(
            inputs,
            GenerationConfig(max_new_tokens=200, stop_strings="<|endoftext|>"),
            tokenizer=processor.tokenizer,
        )

    generated_tokens = output[0, inputs["input_ids"].size(1):]
    generated_text = processor.tokenizer.decode(generated_tokens, skip_special_tokens=True)
    return generated_text


In [ ]:
def run_dataset(dataset_name, processor, model, limit=None):
    """
    Runs Molmo over a dataset, resuming from any existing results file and
    flushing after every row so a Colab disconnect/interrupt loses at most
    the row in progress, not everything since the last full save. Mirrors
    run_dataset() in the Kosmos-2 notebook -- same resume logic, same
    fieldnames (pred_bbox_* is always blank here since Molmo only points).
    """
    sample_dir = SAMPLES_ROOT / dataset_name
    manifest_path = sample_dir / f"manifest_{dataset_name}.csv"
    if not manifest_path.exists():
        raise FileNotFoundError(f"Manifest not found:\n{manifest_path}")

    with manifest_path.open() as f:
        rows = list(csv.DictReader(f))
    if limit is not None:
        rows = rows[:limit]

    out_path = RESULTS_ROOT / f"results_molmoe_{dataset_name}.csv"

    fieldnames = [
        "model", "dataset", "sample_id", "prompt_id", "prompt_text", "model_prompt",
        "raw_output", "pred_bbox_xmin", "pred_bbox_ymin", "pred_bbox_xmax", "pred_bbox_ymax",
        "pred_centroid_x", "pred_centroid_y", "pred_side_lr", "pred_side_ud",
        "gt_centroid_x", "gt_centroid_y", "gt_side_lr", "gt_side_ud",
    ]

    # Resume support: skip (sample_id, prompt_id) pairs already saved from a
    # previous run of this exact dataset, instead of overwriting from scratch.
    done = set()
    file_exists = out_path.exists() and out_path.stat().st_size > 0
    if file_exists:
        with out_path.open() as f:
            for r in csv.DictReader(f):
                done.add((r["sample_id"], r["prompt_id"]))
        print(f"[resume] found {len(done)} already-completed rows in {out_path.name}")

    rows_to_run = [r for r in rows if (r["sample_id"], r["prompt_id"]) not in done]

    print(f"Dataset: {dataset_name}")
    print(f"Total rows: {len(rows)} | already done: {len(done)} | remaining: {len(rows_to_run)}")

    mode = "a" if file_exists else "w"
    with out_path.open(mode, newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if not file_exists:
            writer.writeheader()
            f.flush()

        for i, row in enumerate(rows_to_run, 1):
            img_path = sample_dir / row["image_path"]
            image = Image.open(img_path).convert("RGB")
            img_w, img_h = image.size

            model_prompt = to_molmo_prompt(row["prompt_text"])
            raw_output = run_one(processor, model, image, model_prompt)
            point_pct = parse_point_from_output(raw_output)

            if point_pct is None:
                pred_cx = pred_cy = pred_side_lr = pred_side_ud = ""
            else:
                x_pct, y_pct = point_pct
                pred_cx, pred_cy, pred_side_lr, pred_side_ud = point_pct_to_pixel_and_side(x_pct, y_pct, img_w, img_h)

            writer.writerow({
                "model": "molmoe-1b",
                "dataset": dataset_name,
                "sample_id": row["sample_id"],
                "prompt_id": row["prompt_id"],
                "prompt_text": row["prompt_text"],
                "model_prompt": model_prompt,
                "raw_output": raw_output,
                "pred_bbox_xmin": "",
                "pred_bbox_ymin": "",
                "pred_bbox_xmax": "",
                "pred_bbox_ymax": "",
                "pred_centroid_x": pred_cx,
                "pred_centroid_y": pred_cy,
                "pred_side_lr": pred_side_lr,
                "pred_side_ud": pred_side_ud,
                "gt_centroid_x": row["gt_centroid_x"],
                "gt_centroid_y": row["gt_centroid_y"],
                "gt_side_lr": row["gt_side_lr"],
                "gt_side_ud": row["gt_side_ud"],
            })
            f.flush()  # ensures this row survives a disconnect/interrupt

            status = "no-detection" if point_pct is None else f"{pred_side_lr}/{pred_side_ud}"
            print(f"[{dataset_name}] {i}/{len(rows_to_run)} | {row['sample_id']} | {row['prompt_id']} -> {status}")

    print("\nCompleted!")
    print(f"Results saved to:\n{out_path}")
    return out_path


## Full runs (resumable -- safe to re-run if interrupted)

In [ ]:
result_path = run_dataset(
    dataset_name="lgg",
    processor=processor,
    model=model,
)


In [ ]:
result_path = run_dataset(
    dataset_name="kvasir",
    processor=processor,
    model=model,
)


In [ ]:
result_path = run_dataset(
    dataset_name="isic",
    processor=processor,
    model=model,
)


## Visualize predictions

Saves one combined PNG per sample -- original image | ground truth mask (red) | visual check (gt bbox/centroid + midlines, red) | one prediction panel per prompt P0-P4 (blue point, no box since Molmo only points) -- all side by side, directly to `RESULTS_ROOT/<dataset>/predicted_visual_check/<sample_id>.png`. Nothing is displayed inline; browse the folder afterward to compare every prompt's prediction for a given image at a glance.

In [ ]:
import matplotlib
matplotlib.use("Agg")  # no inline display -- this cell only writes PNG files to disk
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

def visualize_sample_all_prompts(dataset_name, results_csv_path, sample_id):
    """
    Saves (does NOT display) ONE combined PNG per sample_id, with every
    prompt's prediction side by side so they're easy to compare at a glance:
      1. Original image
      2. Ground truth -- the REAL segmentation mask overlaid on the image
         (translucent red) plus its centroid, straight from the dataset's
         mask file (never from the model).
      3. Visual check -- same style as scripts/visualize_ground_truth.py:
         the ground-truth bbox/centroid (red) drawn on the plain image, with
         yellow midline/midheight reference lines and a side_lr/side_ud
         label, so the left/right/upper/lower call can be sanity-checked
         directly against the drawn box instead of the mask overlay.
      4..N. Prediction -- Molmo's predicted point (blue dot) for each
         prompt_id found in results_csv_path for this sample, in prompt_id
         order (P0-P4), one panel per prompt. No box is drawn (Molmo only
         points, see to_molmo_prompt()/parse_point_from_output() above).

    Written to RESULTS_ROOT/<dataset_name>/predicted_visual_check/
    <sample_id>.png -- so you can browse every (input image, ground truth,
    visual check, all-prompts prediction) row as a single file later,
    without needing to re-run any cells or scroll back through notebook
    output.
    """
    sample_dir = SAMPLES_ROOT / dataset_name
    manifest_path = sample_dir / f"manifest_{dataset_name}.csv"

    with manifest_path.open() as f:
        manifest_row = next(r for r in csv.DictReader(f) if r["sample_id"] == sample_id)

    image = Image.open(sample_dir / manifest_row["image_path"]).convert("RGB")
    mask = Image.open(sample_dir / manifest_row["mask_path"]).convert("L")
    img_w, img_h = image.size

    with Path(results_csv_path).open() as f:
        sample_rows = sorted(
            (r for r in csv.DictReader(f) if r["sample_id"] == sample_id),
            key=lambda r: r["prompt_id"],
        )

    n_panels = 3 + len(sample_rows)
    fig, axes = plt.subplots(1, n_panels, figsize=(5 * n_panels, 5))

    # Panel 1: original image, untouched
    axes[0].imshow(image)
    axes[0].set_title(f"{sample_id}\nOriginal image")
    axes[0].axis("off")

    # Panel 2: ground truth -- real mask overlaid + centroid
    axes[1].imshow(image)
    mask_arr = np.array(mask)
    overlay = np.zeros((*mask_arr.shape, 4))
    overlay[mask_arr > 127] = [1, 0, 0, 0.4]  # translucent red where mask is foreground
    axes[1].imshow(overlay)
    gt_cx = float(sample_rows[0]["gt_centroid_x"])
    gt_cy = float(sample_rows[0]["gt_centroid_y"])
    axes[1].plot(gt_cx, gt_cy, "ro", markersize=8)
    axes[1].set_title(f"Ground truth\n{sample_rows[0]['gt_side_lr']}/{sample_rows[0]['gt_side_ud']}")
    axes[1].axis("off")

    # Panel 3: visual check -- gt bbox + centroid + midlines on the plain
    # image, matching scripts/visualize_ground_truth.py's annotation style,
    # so left/right/upper/lower can be sanity-checked against the drawn box.
    axes[2].imshow(image)
    axes[2].axvline(img_w / 2, color="yellow", linewidth=1)
    axes[2].axhline(img_h / 2, color="yellow", linewidth=1)
    gx0 = float(manifest_row["gt_bbox_xmin"])
    gy0 = float(manifest_row["gt_bbox_ymin"])
    gx1 = float(manifest_row["gt_bbox_xmax"])
    gy1 = float(manifest_row["gt_bbox_ymax"])
    axes[2].add_patch(patches.Rectangle((gx0, gy0), gx1 - gx0, gy1 - gy0, linewidth=2, edgecolor="red", facecolor="none"))
    axes[2].plot(gt_cx, gt_cy, "ro", markersize=8)
    axes[2].set_title(f"GT Sanity Check (bbox + midline)\n{sample_rows[0]['gt_side_lr']}/{sample_rows[0]['gt_side_ud']}")
    axes[2].axis("off")

    # Panels 4..N: one per prompt -- point only (Molmo has no bbox; draw a
    # dot whenever pred_centroid_x is present, skip only on no-detection).
    for ax, result_row in zip(axes[3:], sample_rows):
        ax.imshow(image)
        if result_row["pred_centroid_x"]:
            pred_cx, pred_cy = float(result_row["pred_centroid_x"]), float(result_row["pred_centroid_y"])
            ax.plot(pred_cx, pred_cy, "bo", markersize=10)
            pred_label = f"{result_row['pred_side_lr']}/{result_row['pred_side_ud']}"
        else:
            pred_label = "no-detection"
        ax.set_title(f"{result_row['prompt_id']}: {result_row['prompt_text']}\n{pred_label}", fontsize=9)
        ax.axis("off")

    plt.tight_layout()

    viz_dir = RESULTS_ROOT / dataset_name / "predicted_visual_check"
    viz_dir.mkdir(parents=True, exist_ok=True)
    out_file = viz_dir / f"{sample_id}.png"
    plt.savefig(out_file, dpi=100, bbox_inches="tight")
    plt.close(fig)
    return out_file

def visualize_dataset_all_prompts(dataset_name):
    """
    Saves one combined PNG per sample (original | ground truth | visual
    check | every prompt's prediction, side by side) for the WHOLE dataset
    -- no inline display, just files on disk under
    RESULTS_ROOT/<dataset_name>/predicted_visual_check/.
    """
    result_path = RESULTS_ROOT / f"results_molmoe_{dataset_name}.csv"

    with result_path.open() as f:
        sample_ids = sorted(set(r["sample_id"] for r in csv.DictReader(f)))

    for i, sample_id in enumerate(sample_ids, 1):
        out_file = visualize_sample_all_prompts(dataset_name, result_path, sample_id)
        if i % 10 == 0 or i == len(sample_ids):
            print(f"[{dataset_name}] {i}/{len(sample_ids)} | saved -> {out_file}")

    print(f"\nDone. All {len(sample_ids)} samples saved to "
          f"{(RESULTS_ROOT / dataset_name / 'predicted_visual_check')}")

for dataset_name in DATASETS:
    visualize_dataset_all_prompts(dataset_name)
